# Logprob Judge — HuggingFace Endpoints
Calcula P(YES) usando endpoints dedicados de HuggingFace (TGI, OpenAI-compatible).
Los resultados se guardan en `logprob_results.json` para cargarlos en el notebook principal.

Variables requeridas en `.env`:
- `HF_TOKEN` — token de HuggingFace
- `HF_ENDPOINT_70B` + `HF_MODEL_70B` — Llama 3.3 70B
- `HF_ENDPOINT_8B`  + `HF_MODEL_8B`  — Llama 3.1 8B
- `HF_ENDPOINT_QWEN`  + `HF_MODEL_QWEN`  — Qwen3 14B (sandbox)
- `HF_ENDPOINT_GEMMA` + `HF_MODEL_GEMMA` — Gemma 3 12B (sandbox)

In [ ]:
!pip install -q openai python-dotenv

In [ ]:
import json, os
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# override=True fuerza que el .env pise cualquier variable ya cargada en el entorno
env_path = Path(__file__).parent / '.env' if '__file__' in dir() else Path('.env')
load_dotenv(dotenv_path=env_path, override=True)

with open('benchmark.json', encoding='utf-8') as f:
    data = json.load(f)

print(f'Loaded {len(data)} records')

In [ ]:
import traceback

print(f'Working directory: {Path.cwd()}')
env_path = Path('.env')
print(f'.env existe aqui: {env_path.exists()} ({env_path.resolve()})')

HF_TOKEN  = os.getenv('HF_TOKEN', '')
URL_8B    = os.getenv('HF_ENDPOINT_8B', '')
MODEL_8B  = os.getenv('HF_MODEL_8B', '')

# Mostramos suficientes caracteres para confirmar que es el token nuevo
print(f'\nHF_TOKEN (primeros 15): {HF_TOKEN[:15] + "..." if HF_TOKEN else "VACÍO"}')
print(f'HF_ENDPOINT_8B: {URL_8B or "VACÍO"}')
print(f'HF_MODEL_8B:    {MODEL_8B or "VACÍO"}')

if URL_8B and 'REEMPLAZAR' not in URL_8B:
    print(f'\n── Test conexión 8B ──────────────────────────────────────')
    try:
        c = OpenAI(base_url=URL_8B.rstrip('/') + '/v1/', api_key=HF_TOKEN)
        resp = c.chat.completions.create(
            model=MODEL_8B,
            messages=[{'role': 'user', 'content': 'YES or NO?'}],
            max_tokens=1, temperature=1,
            logprobs=True, top_logprobs=5,
        )
        print(f'OK — respuesta: {resp.choices[0].message.content}')
    except Exception:
        traceback.print_exc()

In [ ]:
# ── Configuración de endpoints desde .env ────────────────────────────────────
HF_TOKEN = os.getenv('HF_TOKEN', '')

ENDPOINTS = {
    '70b': {
        'url':      os.getenv('HF_ENDPOINT_70B', ''),
        'model':    os.getenv('HF_MODEL_70B', 'meta-llama/Llama-3.3-70B-Instruct'),
        'no_think': False,
    },
    '8b': {
        'url':      os.getenv('HF_ENDPOINT_8B', ''),
        'model':    os.getenv('HF_MODEL_8B', 'meta-llama/Llama-3.1-8B-Instruct'),
        'no_think': False,
    },
    'qwen': {
        'url':      os.getenv('HF_ENDPOINT_QWEN', ''),
        'model':    os.getenv('HF_MODEL_QWEN', 'Qwen/Qwen3-14B'),
        'no_think': True,   # Qwen3 es thinking model — necesita /no_think
    },
    'gemma': {
        'url':      os.getenv('HF_ENDPOINT_GEMMA', ''),
        'model':    os.getenv('HF_MODEL_GEMMA', 'google/gemma-3-12b-it'),
        'no_think': False,
    },
}

JUDGE_SYSTEM = (
    "You are an evaluation assistant. Given a role definition and an assistant response, "
    "determine whether the response adheres to the role. "
    "Answer ONLY with YES (adherent) or NO (non-adherent)."
)

def make_client(endpoint_url):
    base = endpoint_url.rstrip('/') + '/v1/'
    return OpenAI(base_url=base, api_key=HF_TOKEN)

def is_ready(key):
    cfg = ENDPOINTS[key]
    if not cfg['url'] or 'REEMPLAZAR' in cfg['url']:
        print(f'  [{key}] Endpoint no configurado — saltando')
        return False
    try:
        client = make_client(cfg['url'])
        client.chat.completions.create(
            model=cfg['model'],
            messages=[{'role': 'user', 'content': 'YES or NO?'}],
            max_tokens=1, temperature=1,
            logprobs=True, top_logprobs=5,
        )
        print(f'  [{key}] OK — {cfg["model"]}')
        return True
    except Exception as e:
        print(f'  [{key}] No disponible — {e}')
        return False

print('Verificando endpoints...')
ready = {k: is_ready(k) for k in ENDPOINTS}

In [ ]:
# ── Función logprob — agrega todas las variantes de YES/NO ───────────────────
from scipy.special import logsumexp

YES_TOKENS = {'yes'}
NO_TOKENS  = {'no'}

def judge_logprob(role, assistant, client, model, no_think=False):
    kwargs = dict(
        model=model,
        messages=[
            {'role': 'system', 'content': JUDGE_SYSTEM},
            {'role': 'user',   'content': f'ROLE:\n{role}\n\nASSISTANT RESPONSE:\n{assistant}'},
        ],
        max_tokens=1, temperature=1,
        logprobs=True, top_logprobs=10,
    )
    if no_think:
        # TGI: deshabilita el thinking mode de Qwen3 via chat_template_kwargs
        kwargs['extra_body'] = {'chat_template_kwargs': {'enable_thinking': False}}

    msg = client.chat.completions.create(**kwargs)
    top = msg.choices[0].logprobs.content[0].top_logprobs

    yes_lps, no_lps = [], []
    for t in top:
        norm = t.token.strip().lower()
        if norm in YES_TOKENS:
            yes_lps.append(t.logprob)
        elif norm in NO_TOKENS:
            no_lps.append(t.logprob)

    log_p_yes = logsumexp(yes_lps) if yes_lps else -np.inf
    log_p_no  = logsumexp(no_lps)  if no_lps  else -np.inf

    p_yes = np.exp(log_p_yes)
    p_no  = np.exp(log_p_no)
    denom = p_yes + p_no
    return float(p_yes / denom) if denom > 0 else 0.5

In [ ]:
# ── Smoke test (3 ejemplos por endpoint disponible) ───────────────────────────
for key, ok in ready.items():
    if not ok:
        continue
    cfg      = ENDPOINTS[key]
    client   = make_client(cfg['url'])
    no_think = cfg.get('no_think', False)
    print(f'Smoke test — {cfg["model"]}{"  [no_think]" if no_think else ""}')
    for r in data[:3]:
        score = judge_logprob(r['role'], r['assistant'], client, cfg['model'], no_think)
        print(f'  [{r["label"]}] P(YES) = {score:.3f}')

---
## Sandbox — Qwen3-14B y Gemma 3 12B
Celdas de prueba para los modelos intermedios. Correr independientemente del flujo principal.

In [ ]:
# ── Sandbox: smoke test Qwen3-14B y Gemma 3 12B ──────────────────────────────
import traceback

SANDBOX_KEYS = ['qwen', 'gemma']

for key in SANDBOX_KEYS:
    cfg      = ENDPOINTS[key]
    no_think = cfg.get('no_think', False)

    if not cfg['url'] or 'REEMPLAZAR' in cfg['url']:
        print(f'[{key}] No configurado — completá el .env')
        continue

    print(f'\n── {key} ({cfg["model"]}){"  [no_think]" if no_think else ""} ───────────────────────────────')
    try:
        client = make_client(cfg['url'])

        # Diagnóstico: top tokens del primer token
        diag_kwargs = dict(
            model=cfg['model'],
            messages=[{'role': 'user', 'content': 'YES or NO?'}],
            max_tokens=1, temperature=1,
            logprobs=True, top_logprobs=10,
        )
        if no_think:
            diag_kwargs['extra_body'] = {'chat_template_kwargs': {'enable_thinking': False}}

        resp = client.chat.completions.create(**diag_kwargs)
        top_tokens = [(t.token, round(t.logprob, 3))
                      for t in resp.choices[0].logprobs.content[0].top_logprobs]
        print(f'  Top tokens: {top_tokens}')

        # Smoke test (5 records)
        print(f'  Smoke test:')
        for r in data[:5]:
            score = judge_logprob(r['role'], r['assistant'], client, cfg['model'], no_think)
            print(f'    [{r["label"]:25s}] P(YES) = {score:.3f}')

    except Exception:
        traceback.print_exc()

In [ ]:
# ── Correr sobre el dataset completo ─────────────────────────────────────────
results = {}

for key, ok in ready.items():
    if not ok:
        print(f'Skipping {key} — endpoint no disponible')
        continue
    cfg      = ENDPOINTS[key]
    client   = make_client(cfg['url'])
    no_think = cfg.get('no_think', False)
    print(f'Corriendo {key} ({cfg["model"]}){"  [no_think]" if no_think else ""}...')
    scores = [judge_logprob(r['role'], r['assistant'], client, cfg['model'], no_think) for r in data]
    results[key] = {'model': cfg['model'], 'scores': scores}
    print(f'  Listo. Mean P(YES) = {np.mean(scores):.3f}')

print('\nFin.')

In [ ]:
# ── Guardar resultados ────────────────────────────────────────────────────────
# Carga resultados previos si existen (para no pisar el 8B cuando agregues el 70B)
out_path = Path('logprob_results.json')
if out_path.exists():
    existing = json.loads(out_path.read_text())
    saved = existing.get('results', {})
else:
    saved = {}

saved.update(results)

out = {
    'description': 'LLM logprob judge scores — P(YES) per record',
    'n_records': len(data),
    'results': saved,
}
out_path.write_text(json.dumps(out, indent=2))
print(f'Guardado en logprob_results.json ({len(saved)} modelos)')

# ── Separación por modelo guardado ───────────────────────────────────────────
gold = [1 if d['label'] == 'adherent' else 0 for d in data]
for key, v in saved.items():
    scores = v['scores']
    adh  = np.mean([s for s, g in zip(scores, gold) if g == 1])
    viol = np.mean([s for s, g in zip(scores, gold) if g == 0])
    print(f'  {key} ({v["model"]}): adherent={adh:.3f}  violation={viol:.3f}  sep={adh-viol:+.3f}')